In [1]:
pip install torch torchvision torchaudio



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ----------------------------
# Installation des packages nécessaires
# ----------------------------
!pip install --upgrade pip


!pip install notebook

# Traitement et manipulation d'images
!pip install pillow opencv-python numpy

# Visualisation
!pip install matplotlib

# Lecture des fichiers DICOM
!pip install pydicom

# Évaluation métriques
!pip install scikit-learn

# Optionnel: widgets interactifs dans Jupyter
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.1 MB/s  0:00:011.2 MB/s eta 0:00:01:01
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: dejavu events execute kernel kernelspec l

In [3]:
# Essentials (cellule insérée)
import os, sys, glob, math, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import cv2
import pydicom
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Vérification du device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [14]:
import os

data_dir = "/home/latifa/Téléchargements/imgdatabreastcancer/jpeg"

# Parcours récursif pour lister toutes les images
image_paths = []
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_paths.append(os.path.join(root, file))

print("Nombre d'images trouvées :", len(image_paths))
print("Quelques images :", image_paths[:10])


Nombre d'images trouvées : 7397
Quelques images : ['/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.315743475411279675824136470271131835683/2-203.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.315743475411279675824136470271131835683/1-225.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.129108994012902785219325923011884638559/1-125.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.336291577012427810542728610771005346221/1-165.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.12108033812814607712359451120011752709/1-052.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.198401805612744270313271998910596126739/1-116.jpg', '/home/latifa/Téléchargements/imgdatabreastcancer/jpeg/1.3.6.1.4.1.9590.100.1.2.3062400210351583034640070433614313466/2-205.jpg', '/home/latifa/Téléchargement

In [15]:
class MammoDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        label = self.labels[idx]
        if path.lower().endswith(".dcm"):
            dicom_img = pydicom.dcmread(path)
            img = dicom_img.pixel_array
            img = np.uint8((img - img.min()) / (img.max() - img.min()) * 255)
            img = Image.fromarray(img).convert("L")
        else:
            img = Image.open(path).convert("L")
        
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)




In [17]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Charger ResNet18 pré-entraîné
weights = ResNet18_Weights.DEFAULT  # use default pretrained weights
model = models.resnet18(weights=weights)

# Adapter la dernière couche fully connected pour 2 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)


In [21]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])


In [23]:
from torch.utils.data import DataLoader
from torchvision import transforms

# Assuming image_paths already contains all image file paths
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Your Dataset class (unlabeled version)
from PIL import Image
import numpy as np
import pydicom
from torch.utils.data import Dataset

class MammoDataset(Dataset):
    def __init__(self, img_paths, transform=None):
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        if path.lower().endswith(".dcm"):
            dicom_img = pydicom.dcmread(path)
            img = dicom_img.pixel_array
            img = np.uint8((img - img.min()) / (img.max() - img.min()) * 255)
            img = Image.fromarray(img).convert("L")
        else:
            img = Image.open(path).convert("RGB")  # ResNet expects 3 channels
        
        if self.transform:
            img = self.transform(img)
        return img

# Create dataset and DataLoader
dataset = MammoDataset(img_paths=image_paths, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)  # batch=1 for Grad-CAM


In [25]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.4f}")


ValueError: not enough values to unpack (expected 2, got 1)

In [10]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 3.6 MB/s  0:00:033.6 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [13]:
#test 
import pandas as pd

df = pd.read_csv("/home/latifa/Téléchargements/imgdatabreastcancer/labels.csv")  # adjust path
# assuming CSV has columns: filename,label
labels_dict = dict(zip(df['filename'], df['label']))

labels = [labels_dict[os.path.basename(p)] for p in image_paths]


FileNotFoundError: [Errno 2] No such file or directory: '/home/latifa/Téléchargements/imgdatabreastcancer/labels.csv'

In [11]:
labels = [0]*len(image_paths)  # all zeros, just to check loading


In [12]:
from torch.utils.data import DataLoader
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

dataset = MammoDataset(img_paths=image_paths, labels=labels, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for imgs, lbls in dataloader:
    print("Batch images shape:", imgs.shape)
    print("Batch labels:", lbls)
    break


NameError: name 'MammoDataset' is not defined